## World Bank Data Pull:

#### Tried some more data pulling *requests* codes, but didn't work. 
#### This code gave proper results
- I tried to pull for 2 indicators first. The other 2 indicators are commented out which will be used in the actual bulk pull.
- This is to verify if the code works and pull data in the desired format from the source.

In [7]:
import requests
import pandas as pd

INDICATORS = {
    "rd_gdp": "GB.XPD.RSDV.GD.ZS",
    "gdp_growth": "NY.GDP.MKTP.KD.ZG",
    #"population": "SP.POP.TOTL",
    #"tertiary_enrollment": "SE.TER.ENRR"
}

BASE_URL = "https://api.worldbank.org/v2/country/US/indicator"

records = []

for name, code in INDICATORS.items():
    url = f"{BASE_URL}/{code}"
    
    params = {
        "date": "2005:2024",
        "format": "json",
        "per_page": 100, 
        "page": 1
    }

    try:
        response = requests.get(url, params=params)
        data = response.json()
        if len(data) > 1:
            for item in data[1]:
                if item["value"] is not None:  
                    records.append({
                        "country": item["country"]["value"],
                        "year": int(item["date"]),
                        "indicator": name,
                        "value": item["value"]
                    })
    except Exception as e:
        print(f"Error fetching {name}: {e}")

df = pd.DataFrame(records)

if not df.empty:
    df_pivot = df.pivot(index=['country', 'year'], columns='indicator', values='value').reset_index()
    print(df_pivot.tail())
else:
    print("No data found.")

indicator        country  year  gdp_growth   rd_gdp
15         United States  2020   -2.163029  3.42467
16         United States  2021    6.055053  3.48313
17         United States  2022    2.512375  3.58623
18         United States  2023    2.887556      NaN
19         United States  2024    2.793001      NaN


#### Tested the same code for 3 other countries
- Used the same code with same 2 indicators, just requested for 3 countries by changing the **url format** as guided by the **world bank api documentation**.

In [10]:
import requests
import pandas as pd

INDICATORS = {
    "rd_gdp": "GB.XPD.RSDV.GD.ZS",
    "gdp_growth": "NY.GDP.MKTP.KD.ZG",
    #"population": "SP.POP.TOTL",
    #"tertiary_enrollment": "SE.TER.ENRR"
}

BASE_URL = "https://api.worldbank.org/v2/country/DE;JP;KR/indicator"

records = []

for name, code in INDICATORS.items():
    url = f"{BASE_URL}/{code}"
    
    params = {
        "date": "2005:2022",
        "format": "json",
        "per_page": 100, 
        "page": 1
    }

    try:
        response = requests.get(url, params=params)
        data = response.json()
        if len(data) > 1:
            for item in data[1]:
                if item["value"] is not None:  
                    records.append({
                        "country": item["country"]["value"],
                        "year": int(item["date"]),
                        "indicator": name,
                        "value": item["value"]
                    })
    except Exception as e:
        print(f"Error fetching {name}: {e}")

df = pd.DataFrame(records)

if not df.empty:
    df_pivot = df.pivot(index=['country', 'year'], columns='indicator', values='value').reset_index()
    print(df_pivot.tail())
else:
    print("No data found.")

indicator      country  year  gdp_growth   rd_gdp
55         Korea, Rep.  2020   -0.700242  4.79571
56         Korea, Rep.  2021    4.612968  4.90988
57         Korea, Rep.  2022    2.727565  5.21081
58         Korea, Rep.  2023    1.583015      NaN
59         Korea, Rep.  2024    2.003611      NaN


#### I am convinced that the World Bank data can be pulled from using the code above. 

## OECD Data Pull: 

#### After a lot of browsing through the **OECD Data Explorer** website leading to **OECD Api documentation** page and trying out different methods, each failing as the code couldn't request next pages properly, I spent some more time browsing and found:
- There is a **api url generator** for the selected indicators. I used this method in a previous project as well. So this time it was quite easy to implement. 

In [43]:
import requests
import pandas as pd
from io import StringIO

URL = "https://sdmx.oecd.org/public/rest/data/OECD.STI.STP,DSD_MSTI@DF_MSTI,/.A.B+GV+G+T_RS...?startPeriod=1981&endPeriod=2024&dimensionAtObservation=AllDimensions"

response = requests.get(url)
df = pd.read_csv(StringIO(response.text))
print(df.shape)
print(df.head(20))

(206232, 17)
                              DATAFLOW REF_AREA FREQ   MEASURE UNIT_MEASURE  \
0   OECD.STI.STP:DSD_MSTI@DF_MSTI(1.3)      AUT    A  P_ICTPCT         PATN   
1   OECD.STI.STP:DSD_MSTI@DF_MSTI(1.3)      BEL    A  P_ICTPCT         PATN   
2   OECD.STI.STP:DSD_MSTI@DF_MSTI(1.3)      CHL    A  P_ICTPCT         PATN   
3   OECD.STI.STP:DSD_MSTI@DF_MSTI(1.3)      CHL    A  P_ICTPCT         PATN   
4   OECD.STI.STP:DSD_MSTI@DF_MSTI(1.3)      CHL    A  P_ICTPCT         PATN   
5   OECD.STI.STP:DSD_MSTI@DF_MSTI(1.3)      CHL    A  P_ICTPCT         PATN   
6   OECD.STI.STP:DSD_MSTI@DF_MSTI(1.3)      CHL    A  P_ICTPCT         PATN   
7   OECD.STI.STP:DSD_MSTI@DF_MSTI(1.3)      CHL    A  P_ICTPCT         PATN   
8   OECD.STI.STP:DSD_MSTI@DF_MSTI(1.3)      CHL    A  P_ICTPCT         PATN   
9   OECD.STI.STP:DSD_MSTI@DF_MSTI(1.3)      CHL    A  P_ICTPCT         PATN   
10  OECD.STI.STP:DSD_MSTI@DF_MSTI(1.3)      CHL    A  P_ICTPCT         PATN   
11  OECD.STI.STP:DSD_MSTI@DF_MSTI(1.3) 

/tmp/nix-shell.9EGbym/ipykernel_33389/1642749532.py:8: DtypeWarning: Columns (9,10,11,12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(StringIO(response.text))


- I realized there were many indicators being pulled as the "MSTI" indicator dataset are downloaded in bulk. So, need to filter the data out by keepin the needed 4 indicators.  

In [44]:
print(df['MEASURE'].unique())

['P_ICTPCT' 'P_BIOPCT' 'P_PCT' 'P_TRIAD' 'PNP' 'GV_FB' 'G_FA' 'H_FB'
 'G_FON' 'C_GUF' 'C_SPA' 'C_DF' 'B_FON' 'B_FA' 'B_AERO' 'B_FG' 'C_HEA'
 'C_EDU' 'C_CV' 'C_ECO' 'C_NOR' 'B_FB' 'G_FG' 'H_WRS' 'T_WRS' 'G_CV'
 'T_RS' 'H_RS' 'PPP' 'TD_ECOMP' 'TD_EAERO' 'TD_EDRUG' 'GV' 'B' 'H' 'B_RS'
 'B_WRS' 'G' 'G_TT' 'T_TT' 'G_RS' 'H_TT' 'B_TT' 'G_FB' 'G_BR' 'B_MANUF'
 'B_ICTS' 'B_COMP' 'B_DRUG' 'B_SERV' 'G_WRS' 'TOT_POP' 'TOT_EMP' 'C' 'GDP'
 'TD_IDRUG' 'TD_IAERO' 'TD_BCOMP' 'TD_BAERO' 'TD_ICOMP' 'TD_BDRUG' 'PI']


In [45]:
MY_MEASURES = ['G','B','GV','T_RS']
df_filtered = df[df['MEASURE'].isin(MY_MEASURES)]
print(df_filtered.shape)
print(df_filtered[['REF_AREA', 'MEASURE', 'TIME_PERIOD', 'OBS_VALUE']].head(20))

(38269, 17)
     REF_AREA MEASURE  TIME_PERIOD  OBS_VALUE
1590      SGP    T_RS         2000        NaN
9371      AUS      GV         1981  45.110410
9372      AUS      GV         1984  39.706555
9373      AUS      GV         1986  34.281219
9374      AUS      GV         1987  32.338335
9375      AUS      GV         1988  31.621653
9376      AUS      GV         1990  32.631176
9377      AUS      GV         1992  28.134014
9378      AUS      GV         1994  26.464391
9379      AUS      GV         1996  23.478665
9380      AUS      GV         1998  22.909331
9381      AUS      GV         2000  22.614739
9382      AUS      GV         2002  18.788035
9383      AUS      GV         2004  15.568150
9384      AUS      GV         2006  14.213949
9385      AUS      GV         2008  12.086957
9386      AUT      GV         1981   9.026821
9387      AUT      GV         1985   8.397013
9388      AUT      GV         1989   7.458125
9389      AUT      GV         1998   6.440048


The above data is for the filtered indicators:
- GERD(G)
- BERD(B)
- GOVERD(GV)
- Researchers(T_RS)

In [46]:
df_clean = df_filtered[['REF_AREA', 'MEASURE', 'TIME_PERIOD', 'OBS_VALUE']].copy()
print(df_clean.shape)
print(df_clean.head(20))

(38269, 4)
     REF_AREA MEASURE  TIME_PERIOD  OBS_VALUE
1590      SGP    T_RS         2000        NaN
9371      AUS      GV         1981  45.110410
9372      AUS      GV         1984  39.706555
9373      AUS      GV         1986  34.281219
9374      AUS      GV         1987  32.338335
9375      AUS      GV         1988  31.621653
9376      AUS      GV         1990  32.631176
9377      AUS      GV         1992  28.134014
9378      AUS      GV         1994  26.464391
9379      AUS      GV         1996  23.478665
9380      AUS      GV         1998  22.909331
9381      AUS      GV         2000  22.614739
9382      AUS      GV         2002  18.788035
9383      AUS      GV         2004  15.568150
9384      AUS      GV         2006  14.213949
9385      AUS      GV         2008  12.086957
9386      AUT      GV         1981   9.026821
9387      AUT      GV         1985   8.397013
9388      AUT      GV         1989   7.458125
9389      AUT      GV         1998   6.440048


In [47]:
df_clean.columns = ['Country', 'Indicator', 'Year', 'Value']
print(df_clean.head(5))

     Country Indicator  Year      Value
1590     SGP      T_RS  2000        NaN
9371     AUS        GV  1981  45.110410
9372     AUS        GV  1984  39.706555
9373     AUS        GV  1986  34.281219
9374     AUS        GV  1987  32.338335


In [48]:
print(df_clean['Country'].nunique())
print(df_clean['Year'].min(), df_clean['Year'].max())

49
1981 2024


- The above 3 cell actions lay the ground for data cleaning process.

## USPTO DATA: 

In [49]:
import requests, os

FILES = {
    "g_patent.tsv.zip": "https://s3.amazonaws.com/data.patentsview.org/download/g_patent.tsv.zip",
    "g_inventor_disambiguated.tsv.zip": "https://s3.amazonaws.com/data.patentsview.org/download/g_inventor_disambiguated.tsv.zip",
}

os.makedirs("../raw_data", exist_ok=True)

for fname, url in FILES.items():
    dest = f"../raw_data/{fname}"
    if os.path.exists(dest):
        print(f"Already exists: {dest}")
        continue
    print(f"Downloading {fname}...")
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                f.write(chunk)
    print(f"  Saved  ")

  Saved  
  Saved  


- Used chunking for this file. Because both of the files are more than 1-2 Gb in size.
- The cells below test the data quality and format. 

#### Unzipped using bash: unzip. 

In [50]:
import pandas as pd
df = pd.read_csv("../raw_data/g_patent.tsv", sep = "\t", nrows = 10000, low_memory = False)
print(df.shape)
print(df.dtypes)
print(df.head())

(10000, 8)
patent_id        int64
patent_type     object
patent_date     object
patent_title    object
wipo_kind       object
num_claims       int64
withdrawn        int64
filename        object
dtype: object
   patent_id patent_type patent_date  \
0   10000000     utility  2018-06-19   
1   10000001     utility  2018-06-19   
2   10000002     utility  2018-06-19   
3   10000003     utility  2018-06-19   
4   10000004     utility  2018-06-19   

                                        patent_title wipo_kind  num_claims  \
0  Coherent LADAR using intra-pixel quadrature de...        B2          20   
1  Injection molding machine and mold thickness c...        B2          12   
2  Method for manufacturing polymer film and co-e...        B2           9   
3  Method for producing a container from a thermo...        B2          18   
4  Process of obtaining a double-oriented film, c...        B2           6   

   withdrawn       filename  
0          0  ipg180619.xml  
1          0  ipg1806

In [51]:
print(df.isnull().sum())

patent_id       0
patent_type     0
patent_date     0
patent_title    0
wipo_kind       0
num_claims      0
withdrawn       0
filename        0
dtype: int64


In [52]:
print(df["patent_type"].value_counts())

patent_type
utility    10000
Name: count, dtype: int64


In [53]:
print(df["patent_date"].min(), df["patent_date"].max())

2018-06-19 2018-07-03


In [56]:
print(df["patent_id"].duplicated().sum())

0


#### patent_id is of object type here. location_id has NaN values.

In [66]:
import pandas as pd
inv = pd.read_csv("../raw_data/g_inventor_disambiguated.tsv", sep = "\t", nrows = 10000, low_memory = False)
print(df.shape)
print(df.dtypes)
print(df.head())

(10000, 7)
patent_id                       object
inventor_sequence                int64
inventor_id                     object
disambig_inventor_name_first    object
disambig_inventor_name_last     object
gender_code                     object
location_id                     object
dtype: object
  patent_id  inventor_sequence           inventor_id  \
0  D1006496                  0    fl:we_ln:jiang-149   
1  12029253                  4    fl:ei_ln:baumker-1   
2   6584128                  0    fl:ri_ln:kroeger-1   
3   4789863                  0       fl:th_ln:bush-1   
4  11161990                  1  fl:ma_ln:boudreaux-4   

  disambig_inventor_name_first disambig_inventor_name_last gender_code  \
0                      Wenjing                       Jiang           F   
1                         Eiko                     BÄUMKER           M   
2                      Richard                     Kroeger           M   
3                    Thomas A.                        Bush           

In [67]:
print(inv.isnull().sum())

patent_id                        0
inventor_sequence                0
inventor_id                      0
disambig_inventor_name_first     2
disambig_inventor_name_last      0
gender_code                     17
location_id                     77
dtype: int64


#### No mismatch of datatype for both

In [68]:
print("patents:", df["patent_id"].dtype)
print("inventors:", inv["patent_id"].dtype)

patents: object
inventors: object


In [69]:
print("patents sample:", df["patent_id"].head(3).tolist())
print("inventors sample:", inv["patent_id"].head(3).tolist())

patents sample: ['D1006496', '12029253', '6584128']
inventors sample: ['D1006496', '12029253', '6584128']


#### To check how many unique inventors exist per patent

In [70]:
print(inv.groupby("patent_id")["inventor_id"].count().describe())

count    9995.000000
mean        1.000500
std         0.022362
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         2.000000
Name: inventor_id, dtype: float64


#### inventor_sequence is the position of an inventor on a patent. 

In [71]:
print(inv["inventor_sequence"].value_counts().sort_index())

inventor_sequence
0     3906
1     2447
2     1571
3      917
4      451
5      276
6      147
7      104
8       63
9       32
10      19
11      26
12       5
13       7
14       7
15       4
16       3
17       5
18       3
19       2
20       1
24       1
25       1
37       1
55       1
Name: count, dtype: int64


In [72]:
for chunk in pd.read_csv("../raw_data/g_patent.tsv", sep = "\t", chunksize = 500_000, low_memory = False):
    print(chunk.shape)

(500000, 8)
(500000, 8)
(500000, 8)
(500000, 8)
(500000, 8)
(500000, 8)
(500000, 8)
(500000, 8)
(500000, 8)
(500000, 8)
(500000, 8)
(500000, 8)
(500000, 8)
(500000, 8)
(500000, 8)
(500000, 8)
(500000, 8)
(500000, 8)
(361444, 8)


In [73]:
null_totals = pd.Series(dtype="int64")

for chunk in pd.read_csv("../raw_data/g_patent.tsv", sep="\t", chunksize=500_000, low_memory=False):
    null_totals = null_totals.add(chunk.isnull().sum(), fill_value=0)

print(null_totals)

filename        0.0
num_claims      0.0
patent_date     0.0
patent_id       0.0
patent_title    0.0
patent_type     0.0
wipo_kind       0.0
withdrawn       0.0
dtype: float64


In [74]:
null_totals = pd.Series(dtype="int64")

for chunk in pd.read_csv("../raw_data/g_inventor_disambiguated.tsv", sep="\t", chunksize=500_000, low_memory=False):
    null_totals = null_totals.add(chunk.isnull().sum(), fill_value=0)

print(null_totals)

disambig_inventor_name_first       656.0
disambig_inventor_name_last        517.0
gender_code                      44879.0
inventor_id                          0.0
inventor_sequence                    0.0
location_id                     164884.0
patent_id                            0.0
dtype: float64


In [75]:
import psutil, os

process = psutil.Process(os.getpid())

for i, chunk in enumerate(pd.read_csv("../raw_data/g_inventor_disambiguated.tsv", sep="\t", chunksize=500_000, low_memory=False)):
    mem = process.memory_info().rss / (1024**2)
    print(f"Chunk {i+1}: {chunk.shape[0]:,} rows | RAM used: {mem:.1f} MB")

Chunk 1: 500,000 rows | RAM used: 875.7 MB
Chunk 2: 500,000 rows | RAM used: 818.4 MB
Chunk 3: 500,000 rows | RAM used: 883.1 MB
Chunk 4: 500,000 rows | RAM used: 818.4 MB
Chunk 5: 500,000 rows | RAM used: 883.1 MB
Chunk 6: 500,000 rows | RAM used: 818.4 MB
Chunk 7: 500,000 rows | RAM used: 883.1 MB
Chunk 8: 500,000 rows | RAM used: 818.4 MB
Chunk 9: 500,000 rows | RAM used: 883.1 MB
Chunk 10: 500,000 rows | RAM used: 818.4 MB
Chunk 11: 500,000 rows | RAM used: 883.1 MB
Chunk 12: 500,000 rows | RAM used: 818.4 MB
Chunk 13: 500,000 rows | RAM used: 883.1 MB
Chunk 14: 500,000 rows | RAM used: 818.4 MB
Chunk 15: 500,000 rows | RAM used: 883.1 MB
Chunk 16: 500,000 rows | RAM used: 818.4 MB
Chunk 17: 500,000 rows | RAM used: 883.1 MB
Chunk 18: 500,000 rows | RAM used: 818.4 MB
Chunk 19: 500,000 rows | RAM used: 883.1 MB
Chunk 20: 500,000 rows | RAM used: 818.4 MB
Chunk 21: 500,000 rows | RAM used: 883.1 MB
Chunk 22: 500,000 rows | RAM used: 818.4 MB
Chunk 23: 500,000 rows | RAM used: 883.1 

#### Basic quality checks end and I hereby draw conclusion to data extraction testing layer. Next I'll test out the transformation logics.